In [1]:
#import library
from pandas_gbq import read_gbq
import pandas as pd
import numpy as np
import os
import datetime
import ssl
import logging

In [2]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Get today's date
today = date.today()

# --- Current month period (CY): last completed calendar month ---
# last day of previous month
end_date = date(today.year, today.month, 1) - timedelta(days=1)
# first day of that month
start_date = date(end_date.year, end_date.month, 1)

# --- Prior period (PP): previous calendar month (1 month) ---
pp_end_date = start_date - timedelta(days=1)
pp_start_date = date(pp_end_date.year, pp_end_date.month, 1)

# --- Corresponding periods last year (LY): same calendar months ---
# "CWLY" = same month last year (range)
cwly_start_date = start_date - relativedelta(years=1)
cwly_end_date   = end_date   - relativedelta(years=1)

# "PWLY" = prior-month period last year (range)
pwly_start_date = pp_start_date - relativedelta(years=1)
pwly_end_date   = pp_end_date   - relativedelta(years=1)

# --- YTD helpers ---
ytd_last_year = end_date - relativedelta(years=1)  # same day last year as this period end
begin_of_current_year = date(end_date.year, 1, 1)
begin_of_last_year    = date(end_date.year - 1, 1, 1)

# Output all dates (I return ranges for LY to match month logic)
(
    end_date, start_date,
    pp_end_date, pp_start_date,
    cwly_start_date, cwly_end_date,
    pwly_start_date, pwly_end_date,
    begin_of_last_year, ytd_last_year, begin_of_current_year
)


(datetime.date(2026, 1, 31),
 datetime.date(2026, 1, 1),
 datetime.date(2025, 12, 31),
 datetime.date(2025, 12, 1),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 31),
 datetime.date(2024, 12, 1),
 datetime.date(2024, 12, 31),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 31),
 datetime.date(2026, 1, 1))

In [ ]:
from custom_query import read_sql_query, sql_files

# Read SQL queries from files
queries = {key: read_sql_query(path) for key, path in sql_files.items()}

# Execute queries if all were successfully reada
if all(queries.values()):
    try:
        monthly_note = read_gbq(queries["weekly_note"], project_id='pcln-pl-airanalytics-prod')
        finance_data = read_gbq(queries["finance_data"], project_id='pcln-pl-airanalytics-prod')
        gds_incentives = read_gbq(queries["gds_incentives"], project_id='pcln-pl-airanalytics-prod')
        tsa_data = read_gbq(queries["tsa_data"], project_id='pcln-pl-airanalytics-prod')
        dau_conversion_data = read_gbq(queries["dau_conversion_query"], project_id='pcln-pl-airanalytics-prod')
        print("SQL queries executed successfully.")
        
    except Exception as e:
        print(f"Failed to execute SQL queries: {e}")

/Users/sye/Library/Python/3.9/lib/python/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=UdfRBVOeDdP0NwnrgCWt78N7rXvuLI&prompt=consent&access_type=offline
Downloading:  14%|█▎        |

## make a copy of dataset

In [64]:
# make a copy of every data set
df_monthly=monthly_note.copy()
df_finance=finance_data.copy()
df_gds_incentive=gds_incentives.copy()
df_tsa=tsa_data.copy()
df_monthly.columns = df_monthly.columns.str.lower()
dau_conversion=dau_conversion_data.copy()


## Summary Table

In [65]:

def format_number(num):
    if pd.isna(num):
        return ''
    if abs(num) >= 1e6:
        return f"{num/1e6:.1f}M"
    elif abs(num) >= 1e3:
        return f"{num/1e3:.0f}K"
    elif abs(num) < 1e3:
        return "<1K"
    return f"{num:.0f}"


def format_percentage(x, decimals=1, multiply_100=True):
    def fmt(v):
        if pd.isna(v):
            return ''
        v = float(v)
        if multiply_100:
            v *= 1
        return f"{v:.{decimals}f}%"

    if isinstance(x, pd.Series):
        return x.map(fmt)
    if isinstance(x, pd.DataFrame):
        return x.applymap(fmt) 
    return fmt(x)  


def format_percentage_2(num):
    if pd.isna(num):
        return ''
    return f"{num:.2f}%"

def round_to_nearest_10(num):
  return round(num / 10) * 10

df_pricelince=df_monthly[(df_monthly['brand']== 'Priceline')]
df_pricelince_air=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['offer_type']== 'Flights Only')]
df_pricelince_b2c=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['company']== 'Priceline B2C')]
df_pricelince_b2c_standalone=df_monthly[(df_monthly['brand']== 'Priceline')]


In [66]:
import kpi_month
import importlib

importlib.reload(kpi_month)

from kpi_month import calculate_business_metrics

# Calculate business metrics
df_business = calculate_business_metrics(df_pricelince,start_date,end_date, pp_start_date,pp_end_date,format_number)


from kpi_month import calculate_carrier_metrics
# Calculate carrier metrics
df_carrier = calculate_carrier_metrics(
    df_pricelince_b2c_standalone,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    format_number,
)


from kpi_month import calculate_channel_metrics
df_channel = calculate_channel_metrics(
    df_pricelince_b2c_standalone,
    start_date, end_date,
    pp_start_date, pp_end_date,
    format_number,
)

In [ ]:
df_business

In [ ]:
df_channel

In [ ]:
df_carrier

##  DAU


In [ ]:
import importlib
import dau_roi_table_month

importlib.reload(dau_roi_table_month)

from dau_roi_table_month import calculate_dau_conversion

df_dau_conversion = calculate_dau_conversion(
    dau_conversion,
    format_percentage,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    cwly_start_date,
    cwly_end_date,
    pwly_start_date,
    pwly_end_date,
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year)

df_dau_conversion

# Summary Table

## YTD

In [ ]:
00
import importlib
import summary_table_actual_month  

importlib.reload(summary_table_actual_month)

from summary_table_actual_month import create_finance_number_month


finance_number = create_finance_number_month(
    df_monthly,
    df_gds_incentive,
    start_date, end_date,                 
    pp_start_date, pp_end_date,           
    cwly_start_date, cwly_end_date,       
    pwly_start_date, pwly_end_date,       
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    format_percentage,
    format_number,
)

finance_number

In [72]:
#current week finance data
#net tickets by scenario
plan_net_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)

#previous week finance data
#net tickets by scenario
plan_net_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


#ytd week finance data
#net tickets by scenario
plan_net_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)



# #ytd last year finance data
# #net tickets by scenario
# plan_net_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
# #gross tickets by scenario
# plan_gr_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
# #net revenue by scenario
# plan_grrev_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
# #net revenue by scenario
# plan_netrev_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


In [ ]:
period_order = ["CW", "PW", "YTD", "YTD_LY"]

# revenue as Series (no squeeze)
plan_grrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= end_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= start_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

# 1) MultiIndex columns (metric, period)
plan_metrics_mi = pd.concat(
    {
        ("net_tkts", "CW"): plan_net_cw,
        ("net_tkts", "PW"): plan_net_pw,
        ("net_tkts", "YTD"): plan_net_ytd,
        # ("net_tkts", "YTD_LY"): plan_net_ytd_ly,

        ("gr_tkts", "CW"): plan_gr_cw,
        ("gr_tkts", "PW"): plan_gr_pw,
        ("gr_tkts", "YTD"): plan_gr_ytd,
        # ("gr_tkts", "YTD_LY"): plan_gr_ytd_ly,

        ("gr_rev", "CW"): plan_grrev_cw,
        ("gr_rev", "PW"): plan_grrev_pw,
        ("gr_rev", "YTD"): plan_grrev_ytd,
        # ("gr_rev", "YTD_LY"): plan_grrev_ytd_ly,

        ("net_rev", "CW"): plan_netrev_cw,
        ("net_rev", "PW"): plan_netrev_pw,
        ("net_rev", "YTD"): plan_netrev_ytd,
        # ("net_rev", "YTD_LY"): plan_netrev_ytd_ly,
    },
    axis=1
)
plan_metrics_mi.index.name = "scenario"

# 2) reshape to "period columns"
plan_metrics_period_cols = (
    plan_metrics_mi
      .stack(0)                # stack metric level -> rows
      .reset_index()
      .rename(columns={"level_1": "metric"})
)

# optional formatting
if callable(format_number):
    for c in period_order:
        if c in plan_metrics_period_cols.columns:
            plan_metrics_period_cols[c] = plan_metrics_period_cols[c].round(0)

keep = ["scenario", "metric"] + [c for c in period_order if c in plan_metrics_period_cols.columns]
plan_metrics_period = plan_metrics_period_cols[keep]

# filter PLAN
plan_metrics_period_plan = plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN")]
plan_metrics_period_plan

In [74]:
def build_vs_plan(finance_number: pd.DataFrame, plan_metrics_period: pd.DataFrame,format_percentage) -> pd.DataFrame:
    plan = (plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN"),
                                    ["metric", "CW", "PW", "YTD"]]
            .set_index("metric")
            .rename(columns={"CW": "CW_plan", "PW": "PW_plan", "YTD": "YTD_plan"}))

    actual = (finance_number[["Measure", "CW", "PW", "CY"]]
              .set_index("Measure"))

    measure_to_metric = {
        "Net Tickets": "net_tkts",
        "Gross Tickets": "gr_tkts",
        "Net Cont + Fee + Incentives + vcc rebate(Flight Only)": "net_rev",
        "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)": "gr_rev",
    }

    tmp = actual.rename(index=measure_to_metric)
    tmp = tmp.join(plan, how="right")
    
    metric_order = list(measure_to_metric.values())
    tmp = tmp.reindex(metric_order)
    def vs_pct(a, p):
        p = p.replace(0, np.nan)
        return (a / p - 1) * 100

    out = pd.DataFrame(index=tmp.index)
    out["Reporting Week (vs Plan)"] = vs_pct(tmp["CW"], tmp["CW_plan"])
    out["Previous Week (vs Plan)"]  = vs_pct(tmp["PW"], tmp["PW_plan"])
    out["YTD (vs Plan)"]            = vs_pct(tmp["CY"], tmp["YTD_plan"])  # CY = YTD actual

    metric_to_measure = {v: k for k, v in measure_to_metric.items()}
    out.index = out.index.map(metric_to_measure)

    for c in ["Reporting Week (vs Plan)", "Previous Week (vs Plan)", "YTD (vs Plan)"]:
        out[c] = out[c].apply(format_percentage)

    return out


In [ ]:
finance_number

In [ ]:
plan_metrics_period

In [ ]:
df_vs_plan = build_vs_plan(finance_number, plan_metrics_period, format_percentage)
df_vs_plan

In [ ]:
finance_number
wanted = [
    "Net Tickets",
    "Gross Tickets",
    "Net Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Normalized Net Tickets",
    "Normalized Gross Tickets",
]

finance_output = finance_number.loc[
    finance_number["Measure"].isin(wanted),
    ["Measure", "CW", "Reporting Week", "Previous Week", "YTD"]
].reset_index(drop=True)

# --- formatting ---
# numeric columns
for c in ["CW"]:
    if c in finance_output.columns:
        finance_output[c] = finance_output[c].apply(format_number)
        
finance_output=finance_output.set_index('Measure')
finance_output=finance_output.rename(columns={"CW": "Actual"})
finance_output=finance_output.reindex(wanted).fillna('')
finance_output


## TSA

In [ ]:
tsa_cy=df_tsa[df_tsa['wk_ending']==end_date]['tsa_passengers'].sum()
pcln_cy=df_tsa[df_tsa['wk_ending']==end_date]['pcln_passengers'].sum()
tsa_actual_cy=(pcln_cy*100/tsa_cy)


tsa_pw=df_tsa[df_tsa['wk_ending']==pp_end_date]['tsa_passengers'].sum()
pcln_pw=df_tsa[df_tsa['wk_ending']==pp_end_date]['pcln_passengers'].sum()
tsa_actual_pw=(pcln_pw*100/tsa_pw)


tsa_cwly=df_tsa[df_tsa['wk_ending']==cwly_date]['tsa_passengers'].sum()
pcln_cwly=df_tsa[df_tsa['wk_ending']==cwly_date]['pcln_passengers'].sum()
tsa_actual_cwly=(pcln_cwly*100/tsa_cwly)



tsa_pwly=df_tsa[df_tsa['wk_ending']==pwly_date]['tsa_passengers'].sum()
pcln_pwly=df_tsa[df_tsa['wk_ending']==pwly_date]['pcln_passengers'].sum()
tsa_actual_pwly=(pcln_pwly*100/tsa_pwly)


tsa_ytd=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_current_year)]['tsa_passengers'].sum()
pcln_ytd=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_current_year)]['pcln_passengers'].sum()
tsa_actual_ytd=round((pcln_ytd*100/tsa_ytd),2)


tsa_ytd_ly=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_last_year)]['tsa_passengers'].sum()
pcln_ytd_ly=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_last_year)]['pcln_passengers'].sum()
tsa_actual_ytd_ly=round((pcln_ytd_ly*100/tsa_ytd_ly),2)

print("TSA CY",tsa_cy,"PCLN CY",pcln_cy,"%TSA market share",tsa_actual_cy)
print("TSA LY",tsa_cwly,"PCLN LY",pcln_cwly,"%TSAmarket share",tsa_actual_cwly)
print("TSA PW",tsa_pw,"PCLN PW",pcln_pw,"%TSAmarket share",tsa_actual_pw)
print("TSA PWLY",tsa_pwly,"PCLN PWly",pcln_pwly,"%TSAmarket share",tsa_actual_pwly)
print("TSA YTD CY",tsa_ytd,"PCLN YTD CY",pcln_ytd,"%TSAmarket share CY YTD ",tsa_actual_ytd)
print("TSA YTD LY",tsa_ytd_ly,"PCLN YTD LY",pcln_cwly,"%TSA market share LY YTD ",tsa_actual_ytd_ly)

In [ ]:
# df_summary=df_summary.reset_index(names=['Metric'])


df_summary= pd.DataFrame()
df_summary.loc['DAU','Actual']=df_roi_v.iloc[:, 0].loc['Total']
df_summary.loc['DAU','Reporting Week']=df_roi_v.iloc[:, 1].loc['Total']
df_summary.loc['DAU','Previous Week']=df_roi_v.iloc[:, 2].loc['Total']
df_summary.loc['DAU','YTD']=df_dau_converison.loc[:, "DAU YoY_YTD"].loc['Total']


df_summary.loc['TSA','Actual']=format_percentage_2(tsa_actual_cy)
df_summary.loc['TSA','Reporting Week']=format_percentage(((tsa_actual_cy/tsa_actual_cwly)-1)*100)
df_summary.loc['TSA','Previous Week']=format_percentage((tsa_actual_pw/tsa_actual_pwly-1)*100)
df_summary.loc['TSA','YTD']=format_percentage((tsa_actual_ytd/tsa_actual_ytd_ly-1)*100)

df_summary=df_summary.fillna('')

df_summary

In [ ]:
final_summary = pd.concat([finance_output,df_summary])
final_summary = final_summary.join(df_vs_plan, how="left").fillna('')
final_summary